# 🟩 Pattern 6 — Multi-Agent Systems

> **One-line definition:** multiple decision-makers, each with its own tools and prompt,
> coordinating to solve one problem.

---

## 1. Why bother? (the honest answer)

**Not** because "more agents = smarter". Because of three concrete limits of one agent:

| Problem with a single agent | Multi-agent fix |
|---|---|
| 30 tools → the LLM picks the wrong one | each agent gets 3–5 tools |
| One prompt must cover every domain | one focused prompt per agent |
| Steps run sequentially | independent agents run **in parallel** |
| One team owns the whole prompt | teams own separate agents |

⚠️ **Cost:** more LLM calls, more latency, more failure modes.
**Rule:** don't reach for multi-agent until a single ReAct agent has actually failed you.

---

## 2. The four topologies

```
① SUPERVISOR                    ② NETWORK (swarm)
   ┌──────────┐                     A ◄──► B
   │supervisor│                     ▲ ╲   ╱ ▲
   └──────────┘                     │  ╳  │
    ╱    │    ╲                     ▼ ╱   ╲ ▼
   A     B     C                    C ◄──► D
  (all report back)              (anyone → anyone)

③ HIERARCHICAL                  ④ PIPELINE
     ┌─────┐                      A → B → C → END
     │ top │                     (fixed order, no
     └─────┘                      routing decisions)
      ╱    ╲
  ┌────┐  ┌────┐
  │sup1│  │sup2│
  └────┘  └────┘
   ╱  ╲    ╱  ╲
  A    B  C    D
```

| Topology | Control | Use when | Risk |
|---|---|---|---|
| **Supervisor** | central | 🥇 default choice | supervisor = bottleneck |
| **Network** | none | agents genuinely peer-level | chaos, loops |
| **Hierarchical** | layered | >6 agents | latency stacks up |
| **Pipeline** | fixed | order is known | it's just a chain |

---

## 3. Key Properties (pointwise)

| Property | Multi-agent |
|---|---|
| Decision makers | ✅ **many** — the defining feature |
| Parallelism | ✅ possible (`Send`) |
| Specialisation | ✅ per-agent prompts + tools |
| Cost | 💰💰 highest |
| Debuggability | 🔴 hardest |
| State sharing | 🟡 the hard design question |

---

## 4. The confusing parts (resolved 👇)

### Q1: "Do sub-agents share state, or have their own?"

**This is the central design decision.** Two models:

| | **Shared state** | **Isolated state (subgraph)** |
|---|---|---|
| Sub-agents see | everything | only what you pass in |
| Coupling | tight | loose |
| Context cost | 🔴 grows fast | 🟢 controlled |
| Implementation | same `MessagesState` | separate schema + a translation node |
| Use when | agents genuinely need each other's work | agents are independent specialists |

👉 **Default to shared `messages` for simplicity, isolate when context blows up.**

---

### Q2: "`Command` vs conditional edges — when do I use which?"

| | Conditional edge | `Command` |
|---|---|---|
| Returns | just the next node name | **state update + next node, together** |
| Where | a separate router function | **inside** the node |
| Use for | pure routing | handoffs (update + jump in one move) |

```python
# Conditional edge: two separate concerns
def node(state): return {"next": "researcher"}
def router(state): return state["next"]

# Command: one atomic move  ⭐ preferred for handoffs
def node(state) -> Command[Literal["researcher"]]:
    return Command(goto="researcher", update={"messages": [...]})
```

⚠️ You must annotate the return type (`Command[Literal[...]]`) or LangGraph can't draw
the graph — it has no other way to know where the node can jump to.

---

### Q3: "How does parallel fan-out actually work? `Send` looks like magic."

`Send(node_name, state)` = "run `node_name` with **this custom state**, right now, in parallel."

```python
def fan_out(state):
    return [Send("worker", {"topic": t}) for t in state["topics"]]
    #      ^ a LIST of Sends -> N parallel copies of `worker`
```

Two rules that trip everyone up:
1. Each `Send` gets **its own private state** — not the parent's state.
2. The target node's results **must** merge via a reducer (`operator.add`), or they overwrite
   each other and you silently lose all but one.

---

### Q4: "My supervisor loops forever between two agents."

Classic. Fixes, in order:

1. Add a **`Literal` type** on the supervisor's routing options — restricts the choices.
2. Track **which agents already ran** in state; tell the supervisor so.
3. Add a **step counter** with a hard cap.
4. Set `recursion_limit`.
5. Make `FINISH` an explicit, well-described option so the supervisor knows how to stop.

---

### Q5: "Isn't a supervisor just a router?"

Almost — but a router runs **once**; a supervisor runs **after every worker**, re-deciding
each time. That loop is what makes it agentic rather than a switch statement.

---

## 5. What we'll build

- **Part A** — Supervisor (the default; build this first)
- **Part B** — `Command`-based handoffs (the network/swarm style)
- **Part C** — Parallel fan-out with `Send` (map-reduce)


## 0. Setup

**Install once:**

```bash
pip install langgraph langchain-openai langchain-core
```

**Set your API key** (any chat model works — swap the import if you use Anthropic/Ollama).


In [ ]:
# --- Standard setup used by every notebook in this series ---
from dotenv import load_dotenv

load_dotenv()  # read .env if present

# def _set(var: str):
#     """Prompt for a key only if it's not already in the environment."""
#     if not os.environ.get(var):
#         os.environ[var] = getpass.getpass(f"{var}: ")
#
#
# _set("GROQ_API_KEY")

from langchain_groq import ChatGroq

# temperature=0 -> deterministic-ish output, easier to reason about while learning
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)
print("LLM ready")

---

# 🟦 PART A — Supervisor

## 6. Step 1 — Specialist agents

**Design rule:** each agent gets a **narrow prompt** and **few tools**. That focus is the
entire point — if you give one agent all the tools, you've built a slow single agent.


In [ ]:

from langchain_core.tools import tool
from langchain.agents import create_agent


# --- Researcher's tools ---
@tool
def web_search(query: str) -> str:
    """Search the web for factual information."""
    return (f"[results for '{query}'] Kafka handles ~1M msgs/sec per broker; "
            "LinkedIn processes 7 trillion msgs/day; p99 latency ~5ms.")


# --- Analyst's tools ---
@tool
def calculator(expression: str) -> str:
    """Evaluate an arithmetic expression, e.g. '7e12 / 86400'."""
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error: {e}"


research_agent = create_agent(
    model=llm.bind_tools([web_search]),
    tools=[web_search],
    system_prompt="You are a researcher. Find facts using web_search. Report findings only — do NOT analyse or write prose. You only have one tool, i.e. web_search.",
)

analyst_agent = create_agent(
    model=llm.bind_tools([calculator]),
    tools=[calculator],
    system_prompt="You are a data analyst. Do calculations on the facts already gathered. "
                  "Report numbers only. You only have one tool, i.e. calculator.",
)

# --- Writer has NO tools — deliberately. It only composes. ---
writer_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt="You are a technical writer. Turn the gathered facts and numbers into a clear 2-paragraph summary. Do not invent new facts. You have no tools.",
)
print("3 specialists ready")

---

## 7. Step 2 — State + supervisor schema

**Note `next`:** it's a plain `str` with **no reducer** — each supervisor decision should
*replace* the previous one, not append.


In [ ]:
from typing import Literal, List, Annotated, TypedDict
from pydantic import BaseModel, Field
from langgraph.graph import MessagesState

MEMBERS = ["researcher", "analyst", "writer"]
OPTIONS = MEMBERS + ["FINISH"]


class Router(BaseModel):
    """The supervisor's decision. Literal restricts it to valid choices ONLY —
    this alone prevents a whole class of hallucinated-agent-name bugs."""
    next: Literal["researcher", "analyst", "writer", "FINISH"] = Field(
        description="Which worker acts next. FINISH when the task is complete."
    )
    reason: str = Field(description="One short sentence explaining the choice.")


class TeamState(MessagesState):
    """Shared state. All agents read/write the same `messages` list."""
    next: str  # no reducer -> replaced each turn
    completed: Annotated[List[str], lambda a, b: a + b]  # accumulates -> loop guard

---

## 8. Step 3 — The supervisor node

**Loop prevention lives here.** Note how `completed` is fed back into the prompt.


In [ ]:
# from langchain_core.messages import HumanMessage, AIMessage
#
# SUPERVISOR_PROMPT = (
#     "You are a supervisor managing these workers: {members}.\n"
#     "Given the conversation, decide who acts NEXT.\n\n"
#     "Typical order: researcher (gather facts) -> analyst (compute) -> writer (compose).\n"
#     "Workers already used: {completed}\n\n"
#     "⚠️ Do NOT re-invoke a worker that already did its job.\n"
#     "Respond FINISH once the writer has produced the summary."
# )
#
# supervisor_llm = llm.with_structured_output(Router)
#
#
# def supervisor_node(state: TeamState) -> dict:
#     done = state.get("completed", [])
#     system = SUPERVISOR_PROMPT.format(members=", ".join(MEMBERS),
#                                       completed=", ".join(done) or "none")
#
#     decision = supervisor_llm.invoke(
#         [{"role": "system", "content": system}] + state["messages"]
#     )
#     print(f"\n🧭 SUPERVISOR -> {decision.next}  ({decision.reason})")
#     return {"next": decision.next}

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.prompts.chat import ChatPromptTemplate

supervisor_prompt_template = ChatPromptTemplate.from_messages(
    ("system", """
        You are a supervisor managing these workers: {members}.\n
        Given the conversation, decide who acts NEXT.\n\n
        Typical order: researcher (gather facts) -> analyst (compute) -> writer (compose).\n
        Workers already used: {completed}\n\n
        ⚠️ Do NOT re-invoke a worker that already did its job.\n
        Respond FINISH once the writer has produced the summary.

        You have only

    """)
)

supervisor_llm = llm.with_structured_output(Router)


def supervisor_node(state: TeamState) -> dict:
    done = state.get("completed", [])

    system_prompt = supervisor_prompt_template.format_prompt(
        members=", ".join(MEMBERS),
        completed=", ".join(done) or "none"
    )

    decision = supervisor_llm.invoke(
        [SystemMessage(content=system_prompt.to_string())] + state["messages"]
    )

    print(f"\n🧭 SUPERVISOR -> {decision.next}  ({decision.reason})")
    return {"next": decision.next}

### Worker wrapper

**Why wrap?** Two reasons:
1. The sub-agent returns its own full message list — we only want its **final answer**.
2. We relabel it as an `AIMessage` with a `name`, so the supervisor can tell who said what.


In [ ]:
def make_worker(agent, name: str):
    """Factory: wraps a ReAct sub-agent as a single node in the parent graph."""

    def node(state: TeamState) -> dict:
        print(f"  🤖 {name} working...")
        last_state = {}
        for chunk in agent.stream({"messages": state["messages"]}, stream_mode="values"):
            last_state = chunk
            print("\n" + "*" * 52 + "\n")
            print(last_state["messages"][-1].pretty_print())


        # ⭐ Only the LAST message goes back to the parent.
        # Returning the sub-agent's whole history would explode the context.
        final = last_state["messages"][-1].content
        return {
            "messages": [AIMessage(content=final, name=name)],
            "completed": [name],  # reducer appends -> supervisor sees the history
        }

    return node


researcher_node = make_worker(research_agent, "researcher")
analyst_node = make_worker(analyst_agent, "analyst")
writer_node = make_worker(writer_agent, "writer")

---

## 9. Step 4 — Build the star topology


In [ ]:
from langgraph.graph import StateGraph, START, END

b = StateGraph(TeamState)

b.add_node("supervisor", supervisor_node)
b.add_node("researcher", researcher_node)
b.add_node("analyst", analyst_node)
b.add_node("writer", writer_node)

b.add_edge(START, "supervisor")

# ⭐ Every worker reports BACK to the supervisor. This is what makes it a star.
for m in MEMBERS:
    b.add_edge(m, "supervisor")

# The supervisor fans out. Map FINISH -> END.
b.add_conditional_edges(
    "supervisor",
    lambda s: s["next"],  # read the decision
    {"researcher": "researcher", "analyst": "analyst",
     "writer": "writer", "FINISH": END},  # dict maps value -> node
)

supervisor_graph = b.compile()
print(supervisor_graph.get_graph().draw_mermaid())

In [ ]:
supervisor_graph

---

## 10. Step 5 — Run it


In [ ]:
task = ("Research Kafka's throughput at scale, work out the messages per second "
        "from the daily figure, then write a short summary for an architect.")

out = supervisor_graph.invoke(
    {"messages": [HumanMessage(content=task)], "completed": []},
    config={"recursion_limit": 20},  # ⭐ non-negotiable loop guard
)

print("\n" + "=" * 60)
print("FINAL:\n")
print(out["messages"][-1].content)
print(f"\nWorkers used: {out['completed']}")

---

## 11. The prebuilt shortcut

```bash
pip install langgraph-supervisor
```

Everything above in ~5 lines. Use it in real projects.


In [ ]:
try:
    from langgraph_supervisor import create_supervisor

    prebuilt = create_supervisor(
        agents=[research_agent, analyst_agent, writer_agent],
        model=llm,
        prompt="You manage a researcher, an analyst and a writer. Delegate one at a time.",
    ).compile()
    print("langgraph-supervisor available ✅")
except ImportError:
    print("Not installed. `pip install langgraph-supervisor` to try it.")
    print("Also see: `pip install langgraph-swarm` for the network topology.")

---

# 🟩 PART B — Handoffs with `Command` (network / swarm)

No supervisor. Agents hand off **directly** to each other. Cheaper (no extra routing LLM
call) but easier to get stuck in a loop.


In [ ]:
from langgraph.types import Command


def triage(state: MessagesState) -> Command[Literal["billing_bot", "tech_bot", "__end__"]]:
    """⭐ Command = state update + jump, in ONE atomic return.

    The Literal[...] annotation is REQUIRED — it's the only way LangGraph knows
    which nodes this one can reach, so it can draw and validate the graph.
    """
    text = state["messages"][-1].content.lower()

    if any(w in text for w in ["invoice", "charge", "payment", "refund"]):
        return Command(
            goto="billing_bot",
            update={"messages": [AIMessage(content="Handing off to billing.", name="triage")]},
        )
    if any(w in text for w in ["error", "crash", "bug", "broken"]):
        return Command(
            goto="tech_bot",
            update={"messages": [AIMessage(content="Handing off to tech.", name="triage")]},
        )
    return Command(
        goto="__end__",  # string form of END
        update={"messages": [AIMessage(content="Could you clarify your issue?", name="triage")]},
    )


def billing_bot(state: MessagesState) -> dict:
    return {"messages": [AIMessage(content="Billing: refunded within 5 business days.",
                                   name="billing_bot")]}


def tech_bot(state: MessagesState) -> dict:
    return {"messages": [AIMessage(content="Tech: please send us your error logs.",
                                   name="tech_bot")]}

In [ ]:
hb = StateGraph(MessagesState)
hb.add_node("triage", triage)
hb.add_node("billing_bot", billing_bot)
hb.add_node("tech_bot", tech_bot)

hb.add_edge(START, "triage")
# ⭐ NO conditional edges needed — Command carries the routing itself.
hb.add_edge("billing_bot", END)
hb.add_edge("tech_bot", END)

handoff_graph = hb.compile()

for q in ["I was charged twice", "The app crashes on launch"]:
    r = handoff_graph.invoke({"messages": [HumanMessage(content=q)]})
    print(f"{q!r:35} -> {r['messages'][-1].name}: {r['messages'][-1].content}")

### Handoffs as *tools* (the real swarm pattern)

Give each agent a `transfer_to_X` tool. Now the **LLM** decides the handoff, not `if/else`.


In [ ]:
from langchain_core.tools import tool
from langgraph.types import Command
from langchain_core.messages import ToolMessage
from langchain_core.tools import InjectedToolCallId
from typing import Annotated as Ann
from langgraph.prebuilt import InjectedState


def make_handoff_tool(agent_name: str):
    """Creates a tool that, when called, JUMPS to another agent."""

    @tool(f"transfer_to_{agent_name}",
          description=f"Transfer the conversation to the {agent_name} agent.")
    def handoff(
            state: Ann[dict, InjectedState],  # injected, LLM never sees it
            tool_call_id: Ann[str, InjectedToolCallId],  # injected, LLM never sees it
    ) -> Command:
        # ⚠️ MUST return a ToolMessage or the provider rejects the unanswered tool_call
        msg = ToolMessage(content=f"Transferred to {agent_name}", tool_call_id=tool_call_id)
        return Command(
            goto=agent_name,
            update={"messages": state["messages"] + [msg]},
            graph=Command.PARENT,  # ⭐ jump in the PARENT graph, not this subgraph
        )

    return handoff


print("Injected args (InjectedState / InjectedToolCallId) are hidden from the LLM's schema.")
print("Command.PARENT lets a tool inside a sub-agent move the OUTER graph.")

---

# 🟦 PART C — Parallel fan-out with `Send` (map-reduce)

The only topology that gives you **real speedup**. All branches run concurrently.


In [ ]:
import operator
from langgraph.types import Send


class MapState(TypedDict):
    topic: str
    subtopics: List[str]
    # ⭐ WITHOUT operator.add, parallel workers OVERWRITE each other
    # and you silently keep only one result. This reducer is mandatory.
    summaries: Annotated[List[str], operator.add]
    final: str


class WorkerState(TypedDict):
    """⭐ Each Send gets its OWN private state — NOT the parent's.
    Only the keys you put in the Send are visible here."""
    subtopic: str

In [ ]:
class Subtopics(BaseModel):
    subtopics: List[str] = Field(description="3 distinct, independent subtopics")


def split(state: MapState) -> dict:
    """MAP phase: break the topic into independent chunks."""
    r = llm.with_structured_output(Subtopics).invoke(
        f"Break '{state['topic']}' into exactly 3 independent subtopics."
    )
    print(f"📂 split into: {r.subtopics}")
    return {"subtopics": r.subtopics}


def fan_out(state: MapState):
    """⭐ Returns a LIST of Send objects -> N parallel copies of `research_one`.
    This is NOT a normal router: it returns Sends, not node names."""
    return [Send("research_one", {"subtopic": s}) for s in state["subtopics"]]


def research_one(state: WorkerState) -> dict:
    """Runs in parallel. Sees ONLY {"subtopic": ...} — nothing else from the parent."""
    text = llm.invoke(f"Write 2 concise sentences about: {state['subtopic']}").content
    print(f"  ✓ done: {state['subtopic']}")
    # operator.add on the PARENT's `summaries` merges all N results safely.
    return {"summaries": [f"**{state['subtopic']}**: {text}"]}


def reduce_node(state: MapState) -> dict:
    """REDUCE phase: all parallel results have merged into state['summaries']."""
    joined = "\n\n".join(state["summaries"])
    final = llm.invoke(f"Combine into one coherent overview:\n\n{joined}").content
    return {"final": final}

In [ ]:
mb = StateGraph(MapState)
mb.add_node("split", split)
mb.add_node("research_one", research_one)
mb.add_node("reduce", reduce_node)

mb.add_edge(START, "split")
# ⭐ The fan-out edge. path_map tells LangGraph the Sends target "research_one".
mb.add_conditional_edges("split", fan_out, ["research_one"])
# All parallel branches converge here automatically once they've ALL finished.
mb.add_edge("research_one", "reduce")
mb.add_edge("reduce", END)

map_graph = mb.compile()

res = map_graph.invoke({"topic": "Event-driven architecture with Kafka", "summaries": []})
print("\n" + "=" * 60)
print(res["final"][:900])

---

## 12. Cheat Sheet

```
DEFINITION   multiple decision-makers coordinating on one problem
WHEN         >10 tools | distinct domains | parallelisable work | team ownership
NOT WHEN     a single ReAct agent still works -> multi-agent costs 3-5x

TOPOLOGIES
  Supervisor    star, central control      ← default, start here
  Network       Command handoffs, no boss  ← peers, loop risk
  Hierarchical  supervisor of supervisors  ← >6 agents
  Pipeline      fixed order                ← it's just a chain
  Map-Reduce    Send fan-out               ← the only real speedup

KEY APIS
  Command(goto=..., update=...)          atomic update + jump
  Command[Literal["a","b"]]              REQUIRED return annotation
  Command.PARENT                         jump in the outer graph
  Send("node", {...})                    parallel branch w/ private state
  Annotated[list, operator.add]          MANDATORY for parallel merges
  Literal[...] on Router                 prevents hallucinated agent names
```

**Loop prevention checklist**
- [ ] `Literal` restricts the supervisor's options
- [ ] `completed` list fed back into the supervisor prompt
- [ ] `FINISH` is an explicit, well-described option
- [ ] `recursion_limit` set
- [ ] Only the worker's **last** message returns to the parent

---

## 13. Failure modes

| Symptom | Cause | Fix |
|---|---|---|
| Supervisor ping-pongs | no memory of who ran | track `completed`, feed it into the prompt |
| Context explodes | workers return full history | return only `messages[-1]` |
| Parallel results lost | no reducer on the merge key | `Annotated[list, operator.add]` |
| Hallucinated agent name | free-text routing | `Literal[...]` in the Router schema |
| `Command` node not drawn | missing return annotation | `-> Command[Literal["x"]]` |
| Provider 400 on handoff | tool_call never answered | return a `ToolMessage` with the id |
| Slower than one agent | sequential supervisor | use `Send` for independent work |

---

## 14. Decision Tree

```
Does ONE ReAct agent already work?
├── YES ───────────────────────────► ✅ keep it. Stop here.
└── NO — why not?
    ├── Too many tools (>10)?
    │   └── ► SUPERVISOR (Part A)
    ├── Work is independent & parallelisable?
    │   └── ► SEND / map-reduce (Part C)
    ├── Agents are true peers, no hierarchy?
    │   └── ► COMMAND handoffs / swarm (Part B)
    ├── More than ~6 agents?
    │   └── ► HIERARCHICAL (supervisor of supervisors)
    └── Order is fixed and known?
        └── ► PIPELINE (just add_edge them; not really multi-agent)
```

---

## 15. Next

➡️ **Pattern 7 — Autonomous Goal-Seeking Agents**: give it a goal and walk away.
